# NBA 3.1 - 36 Market-event features (per cohort)

New in NBA 3.1 (no NBA 3.0 equivalent). Spec: `Feature Analysis/92_competitor_comorbidity_nba31_readiness_plan.md` Phase 4 (EV1-EV4). Plain-language summary: `93_competitor_comorbidity_plan_in_simple_words.md`.

**Why:** the NBA model learns from 28 historical cohorts. Two market changes happened inside that window (Entresto generic, Farxiga generic). Without a time-since-event feature the model cannot tell "before" from "after" and mixes the two regimes.

**Columns**

| Column | Definition | Plan ref |
| --- | --- | --- |
| `DAYS_SINCE_FARXIGA_LOE` | `datediff(cohort cutoff, 2026-04-06)`; negative before the event | EV2 |
| `DAYS_SINCE_ENTRESTO_LOE` | `datediff(cohort cutoff, 2025-07-01)`; negative before the event (exact day to confirm with Josh, Q6) | EV2 |
| `FARXIGA_NBRX_LOGRATIO_POST_PRE_LOE` | Per HCP: `ln((post + 0.5) / (pre + 0.5))`, pre = Farxiga brand + generic NBRx (all indications, LRx V7) in the 3 months before the LOE month, post = the LOE month and the 2 after. Empty when the cohort cutoff is before the end of the post window (no future data), or when the HCP has no Farxiga NBRx in either period | EV3 |

**Business rules**
* Cohort cutoff = last day of the cohort's `lookback_end_date` month (same as all other 3.1 features).
* Events live in a reference table (EV1). Only `status = 'build'` events become columns; the others are listed for later NBA versions. Adding an event = adding a row.
* Continuous `DAYS_SINCE_*` is the default; a 0/1 flag is the alternative tested in experiment E4.
* V9: the notebook prints how many cohorts have their cutoff on or after each event (writer counts per cohort still to be added).

Cohorts: `cohorts/lrx` when it exists, else the claims cohorts. Outputs: `hcp_level_model/dev_pipeline/notebook_output/reference/market_events` and `.../hcp_features/market_event_features` (Delta). Widget `TEST_HCP_SAMPLE` > 0 writes `_test` versions.

## Phase 0 validation status (plan 92, Section 3), checked 2026-11-17

**Score: 6 of 10 met, 3 partly met, 1 not met.** Checks that matter for this notebook are in **bold**.

| # | Validation | Result | Status |
| --- | --- | --- | --- |
| V1 | Indication coverage in LAAD | SGLT2 91.5-93.4%, Entresto 100%, steroidal MRA 44.1-49.3%; stable | Met (log-ratio uses all indications) |
| V2 | CKD stage 3 codes present | Event list 11,685,734 stage-3 patients; `Dx_03` 25,167 | Met for 09, not met for `Dx_03` (not used here) |
| V3 | HF LVEF >= 40% proxy sign-off | Medical review not done | Not met (not used here) |
| V4 | CKD indication groups in LRx | CKD, T2D + CKD, T2D + CKD + HF, CKD + HF | Met |
| **V5** | **Farxiga brand vs generic labels** | **Monthly LRx NBRx, brand / generic: Dec 2025 66,996 / 11,491; Jan 2026 54,626 / 43,465; Mar 2026 56,689 / 51,259; Apr 2026 44,465 / 74,977; Jun 2026 18,340 / 130,516. Generic exists from 2023-11-03 and rises from Jan 2026** | **Partly met.** The switch starts before the 2026-04-06 event date, so the pre window (Jan-Mar 2026) already contains generic uptake. The log-ratio adds brand + generic, so it measures total Farxiga momentum, not the switch. Confirm with Josh whether the event date should be Jan 2026 |
| **V6** | **Entresto generic** | **LRx generic: first NBRx 2025-01-31, 541 in Jul 2025, 13,162 in Aug 2025, 27,531 in Sep 2025, while brand fell from 38,518 (Jul) to 15,522 (Sep)** | **Met.** Supports the 2025-07-01 event date; exact day still Q6 |
| V7 | GLP-1 in LRx | 7.01M 12M NBRx | Met (not used here) |
| V8 | Parity vs NBA 3.0 | Proxy only for competitor features | Partly met (not used here) |
| **V9** | **Rows after each event** | **Entresto LOE (cutoff >= 2025-07-01): 14 cohorts (c15-c28), 4,126,178 rows, 86,065 writer-cohorts. Farxiga LOE (cutoff >= 2026-04-06): 5 cohorts (c24-c28), 1,473,635 rows, 32,731 writer-cohorts. Log-ratio filled (cutoff >= 2026-06-30): 3 cohorts (c26-c28), 884,181 rows, 19,772 writer-cohorts. Writers = universe HCPs with Kerendia NBRx in the prediction month** | **Met** for both `DAYS_SINCE_*`. Log-ratio support is thin (3 cohorts). c28 (prediction month Oct 2026, 5,974 writers) is incomplete because LRx ends 2026-10-23 |
| V10 | Entresto prescriber specialty mix | Brand writers CARD 16.1%; Kerendia writers NEPH 19.0% | Met (answered) |

**For this notebook:** V6 and V9 pass. V5 is partly met, so the Farxiga event date needs to be confirmed with Josh.

In [0]:
import os, json, math, builtins
from datetime import datetime as dt
import pandas as pd
import pyspark.sql.functions as F

dbutils.widgets.text("TEST_HCP_SAMPLE", "0", "HCP sample size (0 = full universe)")
TEST_HCP_SAMPLE = int(dbutils.widgets.get("TEST_HCP_SAMPLE"))

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
HCP_UNIVERSE_PATH = f"{VOL31}/HCP_Universe_without_call_hcps"
LRX_TABLE = "`ph-com-ai-us-prod-catalog-2553213839599380`.kerendia_laad_based_lrx.kerendia_laad_based_lrx_v7"
NB_OUT_31 = f"{VOL31}/hcp_level_model/dev_pipeline/notebook_output"
SUFFIX = "_test" if TEST_HCP_SAMPLE else ""
EVENTS_PATH = f"{NB_OUT_31}/reference/market_events{SUFFIX}"
OUTPUT_PATH = f"{NB_OUT_31}/hcp_features/market_event_features{SUFFIX}"
assert all("kerendia_nba_3_1_volume" in p for p in [EVENTS_PATH, OUTPUT_PATH])
COHORTS_DIR = f"{VOL31}/cohorts/lrx" if os.path.exists(f"{VOL31}/cohorts/lrx/metadata.json") else f"{VOL31}/cohorts/claims"

# EV1: event reference table (source: CI report, J. Buffington 9/24/26, and the CI meeting decisions)
# Only status = 'build' becomes a feature. Deferred events have no confirmed date yet.
EVENTS = pd.DataFrame([
    {"event_id": "FARXIGA_LOE",           "event_date": "2026-04-06", "status": "build", "note": "Farxiga generic launch"},
    {"event_id": "ENTRESTO_LOE",          "event_date": "2025-07-01", "status": "build", "note": "Entresto amorphous generic, Jul 2025; exact day to confirm (Q6)"},
    {"event_id": "ENTRESTO_FULL_GENERIC", "event_date": None,         "status": "defer", "note": "Full generic expected 2027"},
    {"event_id": "BAXFENDY_LAUNCH",       "event_date": None,         "status": "defer", "note": "Too few post-event cohorts for NBA 3.1"},
    {"event_id": "EASI_KIDNEY_READOUT",   "event_date": None,         "status": "defer", "note": "Too few post-event cohorts for NBA 3.1"},
    {"event_id": "GLP1_GUIDELINES",       "event_date": None,         "status": "defer", "note": "Too few post-event cohorts for NBA 3.1"},
])
BUILD = EVENTS[EVENTS.status == "build"].reset_index(drop=True)
assert BUILD.event_date.notna().all()

FARXIGA_PGS = ["FARXIGA BRAND", "FARXIGA GENERIC"]
LOGRATIO_COL = "FARXIGA_NBRX_LOGRATIO_POST_PRE_LOE"
FARXIGA_LOE_M = pd.to_datetime(BUILD.set_index("event_id").event_date["FARXIGA_LOE"]).to_period("M").to_timestamp().date()
PRE_START = (pd.Timestamp(FARXIGA_LOE_M) - pd.DateOffset(months=3)).date()     # 2026-01-01
POST_END_M = (pd.Timestamp(FARXIGA_LOE_M) + pd.DateOffset(months=2)).date()    # 2026-06-01 (month of the last post month)
DAYS_COLS = [f"DAYS_SINCE_{e}" for e in BUILD.event_id]
FEATURE_COLS = DAYS_COLS + [LOGRATIO_COL]
print(f"features {FEATURE_COLS} | pre {PRE_START}..{FARXIGA_LOE_M} (excl) | post {FARXIGA_LOE_M}..{POST_END_M} | cohorts {COHORTS_DIR}")

spark.createDataFrame(EVENTS.astype(str).replace("None", None)) \
     .withColumn("event_date", F.to_date("event_date")) \
     .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(EVENTS_PATH)
display(spark.read.format("delta").load(EVENTS_PATH))

In [0]:
cohorts = pd.read_parquet(f"{COHORTS_DIR}/cohorts.parquet")
cohorts = cohorts.sort_values("cohort_id", key=lambda s: s.str[1:].astype(int)).reset_index(drop=True)
with open(f"{COHORTS_DIR}/metadata.json") as f:
    meta = json.load(f)
assert len(cohorts) == meta["cohort_count"], meta

hcp_universe = spark.read.parquet(HCP_UNIVERSE_PATH).select(F.col("PROVIDER_ID").cast("string")).distinct()
if TEST_HCP_SAMPLE:
    hcp_universe = hcp_universe.orderBy(F.xxhash64("PROVIDER_ID")).limit(TEST_HCP_SAMPLE)
hcp_universe = hcp_universe.cache()
n_hcp = hcp_universe.count()

coh = (spark.createDataFrame(cohorts[["cohort_id", "lookback_end_date"]].astype(str))
       .withColumn("SVC_DT_M", F.to_date("lookback_end_date")).drop("lookback_end_date")
       .withColumn("CUTOFF", F.last_day("SVC_DT_M")))
print(f"{len(cohorts)} cohorts | HCPs {n_hcp:,}")

# V9: cohorts on or after each event
cut = pd.to_datetime(cohorts.lookback_end_date) + pd.offsets.MonthEnd(0)
for _, e in BUILD.iterrows():
    n_post = int((cut >= pd.to_datetime(e.event_date)).sum())
    print(f"V9 {e.event_id}: {n_post} of {len(cohorts)} cohorts have the cutoff on or after {e.event_date}")
n_lr = int((cut >= pd.Timestamp(POST_END_M) + pd.offsets.MonthEnd(0)).sum())
print(f"V9 {LOGRATIO_COL}: filled for {n_lr} of {len(cohorts)} cohorts")

In [0]:
# Farxiga (brand + generic, all indications) NBRx per HCP in the pre and post windows
fx = (spark.table(LRX_TABLE)
      .withColumn("M", F.trunc("PERIOD_ID", "month"))
      .filter(F.upper(F.trim("PRODUCT_GROUP")).isin(FARXIGA_PGS)
              & (F.col("M") >= F.lit(str(PRE_START)).cast("date")) & (F.col("M") <= F.lit(str(POST_END_M)).cast("date")))
      .join(F.broadcast(hcp_universe), "PROVIDER_ID", "left_semi")
      .groupBy("PROVIDER_ID").agg(
          F.sum(F.when(F.col("M") < F.lit(str(FARXIGA_LOE_M)).cast("date"), F.col("NBRX")).otherwise(0)).alias("PRE"),
          F.sum(F.when(F.col("M") >= F.lit(str(FARXIGA_LOE_M)).cast("date"), F.col("NBRX")).otherwise(0)).alias("POST")))

post_end_day = F.last_day(F.lit(str(POST_END_M)).cast("date"))
base = hcp_universe.crossJoin(F.broadcast(coh)).join(fx, "PROVIDER_ID", "left")
for _, e in BUILD.iterrows():
    base = base.withColumn(f"DAYS_SINCE_{e.event_id}", F.datediff("CUTOFF", F.lit(e.event_date).cast("date")))
base = base.withColumn(LOGRATIO_COL,
    F.when((F.col("CUTOFF") >= post_end_day) & ((F.coalesce("PRE", F.lit(0)) + F.coalesce("POST", F.lit(0))) > 0),
           F.log((F.coalesce("POST", F.lit(0)) + 0.5) / (F.coalesce("PRE", F.lit(0)) + 0.5))))

market_event_features = (base
    .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "cohort_id"))
    .withColumn("PULL_DT", F.lit(dt.today().strftime("%Y_%m_%d")))
    .select("PROVIDER_ID", "cohort_id", "SVC_DT_M", "HCP_COHORT_ID",
            *[F.col(c).cast("int").alias(c) for c in DAYS_COLS], F.col(LOGRATIO_COL).cast("double"), "PULL_DT"))
market_event_features.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(OUTPUT_PATH)
out = spark.read.format("delta").load(OUTPUT_PATH)
print(f"Saved {out.count():,} rows, {len(out.columns)} columns -> {OUTPUT_PATH}")

In [0]:
n_rows = out.count()
assert n_rows == n_hcp * len(cohorts), (n_rows, n_hcp * len(cohorts))
assert out.select("HCP_COHORT_ID").distinct().count() == n_rows, "HCP_COHORT_ID not unique"

# DAYS_SINCE_* must equal a pandas recompute and be the same for every HCP in a cohort
exp = cohorts[["cohort_id"]].copy()
for _, e in BUILD.iterrows():
    exp[f"DAYS_SINCE_{e.event_id}"] = (cut - pd.to_datetime(e.event_date)).dt.days.values
obs = out.groupBy("cohort_id").agg(*[F.min(c).alias(f"min_{c}") for c in DAYS_COLS], *[F.max(c).alias(f"max_{c}") for c in DAYS_COLS]).toPandas()
chk = exp.merge(obs, on="cohort_id")
bad_days = sum(int(((chk[f"min_{c}"] != chk[c]) | (chk[f"max_{c}"] != chk[c])).sum()) for c in DAYS_COLS)
# no future data: log-ratio empty for every cohort whose cutoff is before the end of the post window
leak = out.join(coh.select("cohort_id", "CUTOFF"), "cohort_id").filter((F.col("CUTOFF") < post_end_day) & F.col(LOGRATIO_COL).isNotNull()).count()
print(f"DAYS_SINCE mismatches: {bad_days} | log-ratio filled before the post window ended: {leak}")
assert bad_days == 0 and leak == 0
display(chk[["cohort_id"] + DAYS_COLS])

last = out.filter(F.col("cohort_id") == cohorts.cohort_id.iloc[-1])
s = last.agg(F.count(LOGRATIO_COL).alias("n"), F.percentile_approx(LOGRATIO_COL, 0.5).alias("med"),
             F.avg(F.when(F.col(LOGRATIO_COL) < 0, 1).otherwise(0)).alias("neg")).collect()[0]
print(f"{cohorts.cohort_id.iloc[-1]}: {LOGRATIO_COL} filled for {s['n']:,} of {n_hcp:,} HCPs, median {s['med']}")

In [0]:
# Output table: one row per HCP x cohort (key HCP_COHORT_ID)
# Reads the saved output, so this cell and the next only need 'Paths, parameters and event reference table' and 'Load cohorts and HCP universe' run first
out = spark.read.format("delta").load(OUTPUT_PATH)
FEATURE_COLS = [c for c in out.columns if c not in {"PROVIDER_ID", "cohort_id", "SVC_DT_M", "HCP_COHORT_ID", "PULL_DT"}]
display(out.select("PROVIDER_ID", "cohort_id", "SVC_DT_M", "HCP_COHORT_ID", *FEATURE_COLS))

In [0]:
# Fill rate per feature: % of rows with a value (not null) and % with a non-zero value,
# over all cohorts and for the latest cohort
latest = cohorts.cohort_id.iloc[-1]

def _fill(df):
    n = df.count()
    r = df.agg(*[F.count(c).alias(f"nn_{c}") for c in FEATURE_COLS],
               *[F.count(F.when(F.col(c) != 0, 1)).alias(f"nz_{c}") for c in FEATURE_COLS]).collect()[0]
    return ([builtins.round(100 * r[f"nn_{c}"] / n, 2) for c in FEATURE_COLS],
            [builtins.round(100 * r[f"nz_{c}"] / n, 2) for c in FEATURE_COLS])

nn_all, nz_all = _fill(out)
nn_last, nz_last = _fill(out.filter(F.col("cohort_id") == latest))
display(pd.DataFrame({"feature": FEATURE_COLS,
                      "pct_filled_all_cohorts": nn_all, "pct_non_zero_all_cohorts": nz_all,
                      f"pct_filled_{latest}": nn_last, f"pct_non_zero_{latest}": nz_last}))

## Feature dictionary, business rules and business logic

**Features created (3)**

| # | Feature | Definition | Level |
| --- | --- | --- | --- |
| 1 | `DAYS_SINCE_FARXIGA_LOE` | Days from the Farxiga generic launch (2026-04-06) to the cohort cutoff; negative before the event | Cohort (same for every HCP) |
| 2 | `DAYS_SINCE_ENTRESTO_LOE` | Days from the Entresto generic launch (2025-07-01) to the cohort cutoff; negative before the event | Cohort (same for every HCP) |
| 3 | `FARXIGA_NBRX_LOGRATIO_POST_PRE_LOE` | `ln((post + 0.5) / (pre + 0.5))`; pre = the HCP's Farxiga (brand + generic) NBRx in 2026-01 to 2026-03, post = 2026-04 to 2026-06 | HCP |

**Business rules**

1. **Events** come from the reference table `notebook_output/reference/market_events` (written by this notebook). Only events with `status = 'build'` become features: Farxiga LOE (2026-04-06) and Entresto LOE (2025-07-01; CI report gives July 2025 only, exact day to confirm with Josh, Q6). Entresto full generic, Baxfendy, EASi-KIDNEY and the GLP-1 guidelines are listed as `defer` (no confirmed date or too few post-event cohorts for NBA 3.1).
2. **Cohort cutoff** = last day of the cohort's `lookback_end_date` month, the same cutoff as all other NBA 3.1 features.
3. **Continuous, signed days:** negative values before the event, so the model can tell pre and post regimes apart and learn how the market settles over time. A 0/1 flag is the alternative tested in experiment E4.
4. **No future data in the log-ratio:** empty for cohorts whose cutoff is before 2026-06-30 (end of the post window); in this build that means c1-c25 are empty and c26-c28 are filled.
5. **Log-ratio null policy:** empty when the HCP has no Farxiga NBRx in either the pre or the post window. The +0.5 keeps the ratio finite when one side is 0.
6. **Source for #3:** LAAD-based LRx V7, `PRODUCT_GROUP` in (FARXIGA BRAND, FARXIGA GENERIC), all indications.
7. **Universe:** every HCP in `HCP_Universe_without_call_hcps` gets one row per cohort (`cohorts/lrx` if it exists, else the claims cohorts).

**Business logic (steps)**

1. Write the event reference table and keep the `build` events.
2. Load cohorts and the universe; print V9 (number of cohorts on or after each event, and cohorts with a filled log-ratio).
3. Sum each universe HCP's Farxiga NBRx in the pre and post windows from LRx V7.
4. Cross-join the universe with all cohorts, compute `DAYS_SINCE_*` = `datediff(cutoff, event_date)` and the log-ratio (only where the cutoff is after the post window), add `HCP_COHORT_ID` and write Delta to `hcp_features/market_event_features`.
5. Validate: row count, unique keys, `DAYS_SINCE_*` equal to a pandas recompute and identical for all HCPs in a cohort, and no log-ratio before the post window ended.